In [1]:
import pandas as pd
import geopandas as geopd

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"]  = 400
plt.style.use('dark_background')


# This script selects and processes flow data for catchments.
# It reads flow data from a Parquet file, clips the data to a specific time period,
# creates virtual measurement stations for bifurcations, and saves the processed data.

In [2]:
# Read the flow data and measurement locations
flows = pd.read_parquet("flow/all_flows.parquet")
places = geopd.read_file('flow/measurement_locations.gpkg', layer="all")

In [6]:
""" CORINE is only available in Finland from 2000 onwards.
If that dataset is used backwards to 1995, we can get 30 years reference period.
I think five years is still quite reasonable, but I'm not really willing to go backwards more.
"""
clipped_flows = flows['1994-01-01':'2023-12-31']
clipped_flows = clipped_flows.dropna(axis=1, how='all')

# Create virtual measurement stations for bifurcations
clipped_flows['6471'] = clipped_flows['2723'] + clipped_flows['3748']
clipped_flows['5847'] = clipped_flows['2687'] + clipped_flows['3160']

In [7]:
# Save the processed flow data to a Parquet file
clipped_flows.to_parquet("flow/flows_from_1994.parquet")

In [45]:
# Filter the measurement locations to include only the clipped flow data columns
clipped_index = clipped_flows.columns
clipped_places = places.loc[places['Paikka_Id'].isin(clipped_index)]

In [50]:
# Save the filtered measurement locations to a GeoPackage
clipped_places.to_file('flow/measurement_locations.gpkg', driver="GPKG", layer="from_1995")

In [ ]:
# Display the clipped flow data and measurement locations
clipped_flows
places